# 03 — Pipeline GPU n°1 : PyTorch → ONNX → TensorRT → CUDA/cuDNN → GPU

| Couche | Technologie |
|---|---|
| Haut niveau | PyTorch, ResNet-18 + tête 10 classes |
| Format du modèle | ONNX (opset 17, batch fixe 10) |
| Middle level | TensorRT : fusion d'opérateurs, kernels spécialisés, FP16 |
| Bas niveau | CUDA / cuDNN |
| Matériel | GPU NVIDIA |

Le notebook exporte le modèle en ONNX, mesure ONNX Runtime CUDA comme contrôle,
puis active l'Execution Provider TensorRT d'ONNX Runtime avec FP16 et cache de
moteur.

In [ ]:
%matplotlib inline
import json
import os
import sys
import time
from pathlib import Path

def _find_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "src" / "common.py").exists():
            return candidate
    return p

ROOT = _find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("TP_ROOT", str(ROOT))
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

IN_COLAB = "google.colab" in sys.modules
REPO_URL = os.environ.get("TP_REPO_URL", "")
if IN_COLAB and REPO_URL and not (ROOT / "src" / "common.py").exists():
    import subprocess
    subprocess.run(["git", "clone", REPO_URL], check=True)
    ROOT = Path(REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")).resolve()
    os.chdir(ROOT)
    sys.path.insert(0, str(ROOT))
    os.environ["TP_ROOT"] = str(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown
from src import common as C

C.ensure_dirs()
C.set_seed()
torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", "4")))
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("Racine du projet :", ROOT)
print("PyTorch :", torch.__version__, "| CUDA :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
print(f"Protocole : {C.N_IMAGES} images | batch={C.BATCH_SIZE} | {C.REPETITIONS} répétitions | warmup={C.WARMUP}")

In [ ]:
def show(result_or_id):
    result = C.load_result(result_or_id) if isinstance(result_or_id, str) else result_or_id
    print(f"{result['label']}  |  device={result['device']}  |  backend={result['backend']}")
    display(pd.DataFrame(C.summary_rows(result)).round(4))
    print(f"Top-1 accuracy = {result['accuracy_top1'] * 100:.2f} %")

def bar_metric(ids, metric, std_metric, ylabel, title, color="#4C72B0"):
    df = pd.DataFrame(C.comparison_rows(ids))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(df["pipeline"], df[metric], yerr=df[std_metric], capsize=5, color=color)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=18)
    for i, value in enumerate(df[metric]):
        if value is not None:
            ax.text(i, value, f"{value:.2f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()
    return df

## 1. Données et modèle

In [ ]:
images, labels = C.load_subset()
x_cpu = C.preprocess_images(images, device="cpu")
print("images :", images.shape, images.dtype)
print("tenseur prétraité :", tuple(x_cpu.shape), x_cpu.dtype)
print("distribution :", np.bincount(labels, minlength=len(C.CLASSES)).tolist())
print("classes :", C.CLASSES)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flat, range(10)):
    ax.imshow(images[idx])
    ax.set_title(C.CLASSES[int(labels[idx])])
    ax.axis("off")
fig.suptitle("Exemples du sous-ensemble CIFAR-10 (5 premières classes)")
plt.tight_layout()
plt.show()

In [ ]:
model = C.build_model("cpu")
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-18 + tête 10 classes : {n_params/1e6:.2f} M paramètres")
print("Couche de classification :", model.fc)
print("Poids de la tête :", C.ARTIFACTS / "classifier_head.npz")

## 2. Export ONNX

In [ ]:
import onnx

onnx_path = C.ARTIFACTS / f"resnet18_cifar10_b{C.BATCH_SIZE}.onnx"
C.export_onnx(model, onnx_path)
onnx.checker.check_model(str(onnx_path))
print("ONNX :", onnx_path, f"({onnx_path.stat().st_size/1e6:.2f} Mo)")

## 3. Vérification du GPU

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU requis. En local : docker compose run --rm tp ... (avec --gpus all). "
        "Sur Colab : Exécution > Modifier le type d'exécution > GPU."
    )
print("GPU :", torch.cuda.get_device_name(0))
print("mémoire :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "Go")
print("CUDA :", torch.version.cuda, "| cuDNN :", torch.backends.cudnn.version())

## 4. Contrôle : ONNX Runtime CUDA

In [ ]:
import onnxruntime as ort

so = ort.SessionOptions()
so.intra_op_num_threads = C.CPU_THREADS
so.inter_op_num_threads = 1
sess_cuda = ort.InferenceSession(
    str(onnx_path),
    sess_options=so,
    providers=[("CUDAExecutionProvider", {"device_id": 0}), "CPUExecutionProvider"],
)
print("providers :", sess_cuda.get_providers())
fn_cuda = C.make_onnx_fn(sess_cuda, sess_cuda.get_inputs()[0].name)
records, summary, logits_cuda = C.run_benchmark(fn_cuda, x_cpu, "cuda")
C.save_result(
    "03a_gpu_onnx_cuda", records, summary, logits_cuda, labels,
    device="cuda", backend="onnxruntime-cuda",
    notes="Contrôle ONNX Runtime CUDA, même graphe ONNX.",
)
show("03a_gpu_onnx_cuda")

## 5. TensorRT (FP16 + cache de moteur)

L'Execution Provider TensorRT construit un moteur optimisé pour le GPU et la
taille de batch. Le premier appel construit le moteur ; le cache permet de
réutiliser le plan aux exécutions suivantes. Les warmups absorbent la
construction avant les mesures.

In [ ]:
trt_cache = C.ARTIFACTS / "trt_cache"
trt_cache.mkdir(parents=True, exist_ok=True)
trt_options = {
    "trt_fp16_enable": True,
    "trt_engine_cache_enable": True,
    "trt_engine_cache_path": str(trt_cache),
    "trt_timing_cache_enable": True,
    "trt_timing_cache_path": str(trt_cache),
    "trt_max_workspace_size": 1 * 1024**3,
    "trt_builder_optimization_level": 3,
}
sess_trt = ort.InferenceSession(
    str(onnx_path),
    sess_options=so,
    providers=[
        ("TensorrtExecutionProvider", trt_options),
        "CUDAExecutionProvider",
        "CPUExecutionProvider",
    ],
)
print("providers :", sess_trt.get_providers())
if "TensorrtExecutionProvider" not in sess_trt.get_providers():
    raise RuntimeError(
        "TensorRT EP non chargé. Vérifiez l'installation de tensorrt et la "
        "compatibilité CUDA. Providers disponibles : "
        + str(sess_trt.get_providers())
    )

input_name = sess_trt.get_inputs()[0].name
x_np = x_cpu.numpy()
t0 = time.perf_counter()
sess_trt.run(None, {input_name: x_np[:C.BATCH_SIZE]})
print(f"construction du moteur + premier run : {time.perf_counter() - t0:.2f} s")

fn_trt = C.make_onnx_fn(sess_trt, input_name)
records, summary, logits_trt = C.run_benchmark(fn_trt, x_cpu, "cuda")
C.save_result(
    "03_gpu_onnx_tensorrt", records, summary, logits_trt, labels,
    device="cuda", backend="tensorrt",
    notes="ONNX -> TensorRT EP, FP16, cache de moteur, workspace 1 Go.",
)
show("03_gpu_onnx_tensorrt")

## 6. Comparaison et intégrité des prédictions

In [ ]:
ref = np.load(C.RESULTS / "00_reference_pytorch_gpu_logits.npy")
for rid in ["03a_gpu_onnx_cuda", "03_gpu_onnx_tensorrt"]:
    r = C.load_result(rid)
    logits_r = np.load(C.RESULTS / f"{rid}_logits.npy")
    print(
        f"{r['label']:<34} Top-1={r['accuracy_top1']*100:5.2f} %  "
        f"accord={C.prediction_agreement(logits_r, ref)*100:5.2f} %"
    )
ids = ["00_reference_pytorch_gpu", "03a_gpu_onnx_cuda", "03_gpu_onnx_tensorrt"]
display(pd.DataFrame(C.comparison_rows(ids)).round(4))
bar_metric(ids, "throughput_ips", "throughput_std", "images/s",
           "Pipeline GPU n°1 : débit", color="#4C72B0")
bar_metric(ids, "latency_ms", "latency_std", "ms / image",
           "Pipeline GPU n°1 : latence", color="#C44E52")

### Interprétation

TensorRT doit améliorer le débit GPU par rapport à ONNX Runtime CUDA grâce à la
fusion d'opérateurs et aux kernels spécialisés. Le mode FP16 peut introduire de
très petites différences numériques ; l'accord de prédictions avec la référence
GPU permet de vérifier que la Top-1 n'est pas dégradée. La mémoire GPU pic et
l'énergie par image complètent l'analyse du compromis performance/coût.